In [1]:
from datasets import load_dataset

# Load your uploaded dataset
dataset = load_dataset("algaebrown/human-kozak-21bp", split = 'train')


README.md:   0%|          | 0.00/309 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  915kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/96346 [00:00<?, ? examples/s]

In [2]:
print(dataset[0], len(dataset[0]['sequence']))

{'id': 'chrom_chr1_pos_65565_strand_+', 'sequence': 'TCTTCCAGTTATGAAGAAGG'} 20


In [3]:
# Vocabulary mapping
NT_MAP = {"A": 0, "C": 1, "G": 2, "T": 3, "*": 4}
MASK_ID = NT_MAP["*"]

# Map text sequences to integer token lists
def encode_sequence(example):
    return {"tokens": [NT_MAP[nt] for nt in example["sequence"]]}


encoded_dataset = dataset.map(encode_sequence)
encoded_dataset.set_format(type="torch", columns=["tokens"])

Map:   0%|          | 0/96346 [00:00<?, ? examples/s]

In [4]:
print(encoded_dataset[0])

{'tokens': tensor([3, 1, 3, 3, 1, 1, 0, 2, 3, 3, 0, 3, 2, 0, 0, 2, 0, 0, 2, 2])}


In [5]:
# Create PyTorch DataLoader
from torch.utils.data import DataLoader
dataloader = DataLoader(encoded_dataset, batch_size=64, shuffle=True)

In [6]:
for x in dataloader:
    print(x['tokens'].shape) # batch_size, L
    break

torch.Size([64, 20])


In [7]:


import torch
import torch.nn as nn


class MaskedDiffusionScheduler(nn.Module):

    def __init__(self, num_timesteps=50, mask_id=4):
        super().__init__()
        self.num_timesteps = num_timesteps
        self.mask_id = mask_id

    def q_sample(self, x_0, t):
        """Forward masking process: q(x_t | x_0)

        x_0: Clean token batch of shape (B, L) (integers) t: Timestep tensor of
        shape (B,) ranging from 1 to num_timesteps
        """
        batch_size, seq_len = x_0.shape
        device = x_0.device

        # Compute masking probability as a function of time t (linear schedule from ~0 to 1)
        # At t=0, mask_prob=0; at t=T, mask_prob approaches 1.0
        mask_probs = t.float() / self.num_timesteps  # Shape: (B,)

        # Expand mask_probs to match sequence shape (B, L)
        # Each sequence in the batch gets its own corruption rate based on its sampled timestep
        mask_probs = mask_probs.unsqueeze(1).expand(-1, seq_len)

        # Generate random uniform matrix to decide which tokens get masked
        rand = torch.rand((batch_size, seq_len), device=device)
        mask_indices = rand < mask_probs  # Boolean mask tensor

        # Copy clean tokens and apply mask where True
        x_t = x_0.clone()
        x_t[mask_indices] = self.mask_id

        return x_t, mask_indices  # Return mask_indices so you can compute loss only on masked positions if desired

    def sample_timesteps(self, batch_size, device="cpu"):
        """Sample random discrete timesteps from 1 to num_timesteps"""
        return torch.randint(
            1, self.num_timesteps + 1, (batch_size,), device=device
        )

In [8]:
# 4. Initialize Scheduler and Device
scheduler = MaskedDiffusionScheduler(num_timesteps=50, mask_id=MASK_ID)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scheduler.to(device)

MaskedDiffusionScheduler()

In [9]:
# 5. Simulation Training Batch Loop
print("\nRunning forward diffusion batch check...")
for batch in dataloader:
    # batch['tokens'] shape: (Batch_Size, 21)
    batch_clean = batch["tokens"].to(device)
    batch_size = batch_clean.shape[0]

    # Sample random timesteps for each sequence in the batch
    t = scheduler.sample_timesteps(batch_size, device=device)

    # Apply the forward masked diffusion process
    batch_noisy, mask_indices = scheduler.q_sample(batch_clean, t)

    # Decode a sample back to letters for inspection
    rev_map = {v: k for k, v in NT_MAP.items()}
    clean_str = "".join([rev_map[token.item()] for token in batch_clean[0]])
    noisy_str = "".join([rev_map[token.item()] for token in batch_noisy[0]])

    print(f"  Timestep (t): {t[0].item()}")
    print(f"  Clean 🧬:   {clean_str}")
    print(f"  Noisy 🛑:   {noisy_str}")
    break


Running forward diffusion batch check...
  Timestep (t): 37
  Clean 🧬:   TCTCACTTAGATGTTGCTGC
  Noisy 🛑:   *****CT*************


In [10]:
batch_noisy.shape

torch.Size([64, 20])

# DiT architecture
https://arxiv.org/pdf/2212.09748

In [11]:
embed_dim = 16

In [12]:
import math
import torch
import torch.nn as nn


class SinusoidalTimeEmbedding(nn.Module):

  def __init__(self, d_model: int):
    """Standard Sinusoidal Time/Position Embedding.

    d_model: The embedding dimension size (must be even).
    """
    super().__init__()
    self.d_model = d_model

    # Compute division term in log space for numerical stability
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
    )

    # Register as buffer so it moves to GPU automatically with model.to(device)
    self.register_buffer("div_term", div_term)

  def forward(self, t: torch.Tensor) -> torch.Tensor:
    """Args:

        t: Tensor of shape (batch_size,) representing continuous or discrete
        timesteps

    Output:
        Tensor of shape (batch_size, d_model)
    """
    batch_size = t.shape[0]
    pe = torch.zeros(batch_size, self.d_model, device=t.device)

    # Broadcast multiplication: [batch_size, 1] * [d_model/2] -> [batch_size, d_model/2]
    scaled_t = t.unsqueeze(-1) * self.div_term

    pe[:, 0::2] = torch.sin(scaled_t)
    pe[:, 1::2] = torch.cos(scaled_t)

    return pe

In [13]:
# s = SinusoidalTimeEmbedding(d_model = 16)
# pe = s(t)
# pe.shape

In [14]:
# import seaborn as sns
# import matplotlib.pyplot as plt
# sns.heatmap(pe.detach().numpy())
# plt.xlabel('d_model')
# plt.ylabel('batch size')


In [15]:
class DiffusionTransformerBlock(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_head, batch_first = True)
        self.layernorm1 = nn.LayerNorm(d_model)
        
        # gamma1, beta1
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(d_model, 2 * d_model))
        self.adaLN_scale = nn.Sequential(nn.SiLU(), nn.Linear(d_model, d_model))

        #
        self.layernorm2 = nn.LayerNorm(d_model)
        # gamma2, beta2
        self.adaLN_modulation2 = nn.Sequential(nn.SiLU(), nn.Linear(d_model, 2 * d_model))
        self.adaLN_scale2 = nn.Sequential(nn.SiLU(), nn.Linear(d_model, d_model))
        self.ff = nn.Sequential(nn.Linear(d_model, 2 * d_model),
                               nn.SiLU(), 
                               nn.Linear(2*d_model,  d_model))

        # qkv projections
        self.k_proj = nn.Linear(d_model, d_model)
        self.q_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
    def forward(self, t, x):
        x_prime = self.layernorm1(x) # B, L, d
        gamma1, beta1 = self.adaLN_modulation(t).chunk(2, dim = -1) # B, d
        
        x_prime = gamma1.unsqueeze(1)*x_prime + beta1.unsqueeze(1)
        x_prime, _ = self.mha(self.q_proj(x_prime), self.k_proj(x_prime), self.v_proj(x_prime))
        
        alpha1 = self.adaLN_scale(t)
        
        x_prime= alpha1.unsqueeze(1)*x_prime +x # scale and residuel connection

        x_prime2 = self.layernorm2(x)
        gamma2, beta2 = self.adaLN_modulation2(t).chunk(2, dim = -1) # B, d
        x_prime2 = gamma2.unsqueeze(1)*x_prime2 + beta2.unsqueeze(1)
        x_prime2 = self.ff(x_prime2)
        alpha2 = self.adaLN_scale(t)
        x_prime2 = alpha2.unsqueeze(1)*x_prime2 +x_prime # scale and residuel connection

        return x_prime2

In [16]:
# d = DiffusionTransformerBlock(16,2)
# fake_x = torch.zeros(64, 20, 16)
# d.forward(pe, fake_x).shape

In [17]:
class PositionalEmbedding(nn.Module):

  def __init__(self, max_len: int, d_model: int):
    """Learnable absolute positional embeddings.

    max_len: Maximum sequence length (e.g., 21 for your Kozak window) d_model:
    The hidden embedding dimension size
    """
    super().__init__()
    self.pos_embedding = nn.Embedding(max_len, d_model)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """Args:

        x: Token embeddings of shape (batch_size, seq_len, d_model)

    Output:
        Tensor of shape (batch_size, seq_len, d_model) with positional info added
    """
    batch_size, seq_len, d_model = x.shape
    device = x.device

    # Generate position indices [0, 1, 2, ..., seq_len - 1]
    positions = torch.arange(seq_len, device=device).unsqueeze(
        0
    )  # Shape: (1, seq_len)

    # Get embeddings for these positions and broadcast across the batch
    pos_embeds = self.pos_embedding(
        positions
    )  # Shape: (1, seq_len, d_model)

    # Element-wise addition to your token embeddings
    return x + pos_embeds

In [18]:

class DiffusionTransformer(nn.Module):
    def __init__(self, d_model, n_heads, n_vocab, max_len):
        super().__init__()
        self.time_embedding = SinusoidalTimeEmbedding(d_model)
        self.token_embedding = nn.Embedding(n_vocab, d_model)
        self.position_embedding = PositionalEmbedding(max_len, d_model)

        # cannot use nn.Sequential because of having two arguments
        self.dits= nn.ModuleList([DiffusionTransformerBlock(d_model,2),
                                  DiffusionTransformerBlock(d_model,2)
                                 ]
                                 )
        self.classifier = nn.Linear(d_model, n_vocab)
    def forward(self, t, x):
        pe = self.time_embedding(t)
        x = self.token_embedding(x)
        
        x = self.position_embedding(x)
        
        for block in self.dits:
          x = block(
              pe, x
          )  # Each block applies attention and time-modulation (AdaLN)
        x = self.classifier(x)
        return torch.softmax(x, dim = -1)
        
        
        

In [19]:
t.shape, batch_noisy.shape,

(torch.Size([64]), torch.Size([64, 20]))

In [20]:
# dit = DiffusionTransformer(16,2,5, 20)
# out_proba = dit(t, batch_noisy)

In [21]:
# mask_indices.shape, out_proba.shape, batch_clean.shape

In [22]:
import torch
import torch.nn.functional as F


def compute_masked_cross_entropy(
    logits: torch.Tensor, targets: torch.Tensor, mask_indices: torch.Tensor
) -> torch.Tensor:
  """Computes cross-entropy loss restricted only to the masked positions.

  Args:
      logits: Raw model predictions of shape (batch_size, seq_len, vocab_size)
      targets: Clean ground-truth token indices of shape (batch_size, seq_len)
      mask_indices: Boolean tensor of shape (batch_size, seq_len) where True
        indicates masked slots.

  Returns:
      A scalar tensor representing the mean cross-entropy loss over masked tokens.
  """
  # 1. Flatten the batch and sequence dimensions
  logits_flat = logits.reshape(-1, logits.shape[-1])
  targets_flat = targets.reshape(-1)
  mask_flat = mask_indices.reshape(-1)

  # 2. Slice out only the positions that were actually corrupted/masked
  masked_logits = logits_flat[mask_flat]
  masked_targets = targets_flat[mask_flat]

  # Safety check: if no positions are masked, return zero loss with gradients attached
  if masked_logits.numel() == 0:
    return torch.tensor(0.0, device=logits.device, requires_grad=True)

  # 3. Compute Cross-Entropy Loss on the filtered subset
  return F.cross_entropy(masked_logits, masked_targets)

In [23]:
import torch
import torch.optim as optim

# 1. Setup Device & Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate model & move to device
# (Note: Using 20 as max_len based on your snippet, or 21 depending on your dataset window)
dit = DiffusionTransformer(
    d_model=16, n_heads=2, n_vocab=5, max_len=20
).to(device)

# 2. Setup Optimizer (AdamW is standard for Transformers and Diffusion)
optimizer = optim.AdamW(dit.parameters(), lr=1e-3, weight_decay=0.01)

# Number of training epochs
num_epochs = 5

print(f"Starting training on device: {device}\n")

dit.train()
for epoch in range(num_epochs):
  total_loss = 0.0
  num_batches = 0

  for batch in dataloader:
    # Get clean tokens and move to device
    batch_clean = batch["tokens"].to(device)  # Shape: (Batch_Size, 20)
    batch_size = batch_clean.shape[0]

    # 1. Sample random timesteps for each sequence in the batch
    t = scheduler.sample_timesteps(batch_size, device=device)

    # 2. Apply the forward masked diffusion process (corruption)
    batch_noisy, mask_indices = scheduler.q_sample(batch_clean, t)

    # 3. Zero out gradients from previous step
    optimizer.zero_grad()

    # 4. Forward pass through the Diffusion Transformer
    # (Ensure forward returns raw logits, NOT softmax probabilities)
    logits = dit(t, batch_noisy)  # Shape: (Batch_Size, 20, 5)

    # 5. Compute masked cross-entropy loss
    loss = compute_masked_cross_entropy(logits, batch_clean, mask_indices)

    # 6. Backpropagation & optimization step
    loss.backward()

    # Optional: Clip gradients to prevent exploding gradients in deep networks
    torch.nn.utils.clip_grad_norm_(dit.parameters(), max_norm=1.0)

    optimizer.step()

    total_loss += loss.item()
    num_batches += 1

  # Print average epoch loss
  avg_loss = total_loss / num_batches
  print(f"Epoch [{epoch+1}/{num_epochs}] | Average Loss: {avg_loss:.4f}")

print("\nTraining completed successfully!")

Starting training on device: cuda

Epoch [1/5] | Average Loss: 1.4354
Epoch [2/5] | Average Loss: 1.4189
Epoch [3/5] | Average Loss: 1.4172
Epoch [4/5] | Average Loss: 1.4159
Epoch [5/5] | Average Loss: 1.4146

Training completed successfully!


In [24]:
import torch
import torch.nn.functional as F


@torch.no_grad()
def sample_kozak_sequences(
    dit,
    num_samples: int = 4,
    seq_len: int = 20,
    num_steps: int = 10,
    device: str = "cpu",
):
  """Generates brand-new Kozak sequences starting from all [MASK] tokens."""
  dit.eval()

  MASK_TOKEN_IDX = 4
  NT_MAP = {0: "A", 1: "C", 2: "G", 3: "T", 4: "[MASK]"}
  rev_map = {v: k for k, v in NT_MAP.items()}

  # 1. Start with everything masked (Batch_Size, Seq_Len)
  current_tokens = torch.full(
      (num_samples, seq_len), MASK_TOKEN_IDX, dtype=torch.long, device=device
  )

  # Create a schedule of timesteps from 1.0 down to near 0.0
  timesteps = torch.linspace(1.0, 0.0, num_steps + 1, device=device)

  print(
      f"Generating {num_samples} sequences over {num_steps} denoising steps..."
  )

  for i in range(num_steps):
    t_start = timesteps[i]
    t_end = timesteps[i + 1]

    # Create a batch tensor filled with the current timestep value
    t_tensor = torch.full((num_samples,), t_start, device=device)

    # 2. Forward pass through the trained model
    logits = dit(t_tensor, current_tokens)  # Shape: (num_samples, seq_len, 5)

    # 3. Prevent the model from predicting a [MASK] during inference
    # Set the logit for index 4 to negative infinity
    logits[:, :, MASK_TOKEN_IDX] = -float("inf")

    # Convert logits to probabilities or take argmax for predictions
    predicted_tokens = torch.argmax(logits, dim=-1)  # (num_samples, seq_len)

    # 4. Update strategy (Simplified linear unmasking schedule):
    # Determine how many tokens should be unmasked at this step based on time t
    # As t drops from 1.0 to 0.0, fewer positions should remain masked.
    mask_prob = t_end.item()

    # Create a random mask for which positions to keep as [MASK] vs fill in
    # (In more advanced implementations, you keep positions with the lowest prediction confidence)
    still_masked = (
        torch.rand(num_samples, seq_len, device=device) < mask_prob
    )

    # If a position is still masked, keep the mask token; otherwise take the model's prediction
    current_tokens = torch.where(
        still_masked,
        torch.tensor(MASK_TOKEN_IDX, device=device),
        predicted_tokens,
    )

  # Final cleanup: ensure zero masks remain in the final output
  current_tokens = torch.where(
      current_tokens == MASK_TOKEN_IDX,
      torch.randint(0, 4, current_tokens.shape, device=device),
      current_tokens,
  )

  # Decode tensor back to string letters for human inspection
  generated_strings = []
  for seq in current_tokens:
    seq_str = "".join([NT_MAP[token.item()] for token in seq])
    generated_strings.append(seq_str)

  return generated_strings


# --- Example Usage ---
generated_seqs = sample_kozak_sequences(dit, num_samples=5, seq_len=20, num_steps=10, device=device)
for idx, seq in enumerate(generated_seqs):
    print(f"Generated Seq {idx+1}: {seq}")

Generating 5 sequences over 10 denoising steps...
Generated Seq 1: CCCCCCCGCCATGGCCCCCC
Generated Seq 2: CCCCCCCGCCATGGCCCCCC
Generated Seq 3: CCCCCCCGCCATGGCCCCCC
Generated Seq 4: CCCCCCCACCATGGCCCCCC
Generated Seq 5: CCCCCCCGCCATGGCCCCCC
